# 💻 大模型工具调用之 CLI 工具实战

在实际工程项目中（比如各种智能编程助手、自动化运维 Agent），大模型不仅可以做简单的 Python API 查询，甚至还可以通过**执行系统 shell 命令行工具**（如 `ls`、`grep`、`find`）来查找和编辑本地文件。

本实验将探讨如何安全地将操作系统命令行（CLI）作为大模型的工具来使用，并强调**防止命令行注入**的安全设计原则。

## 🛠️ 环境初始化：API 密钥与 OpenAI 客户端配置

为了运行本 Notebook，我们需要配置大模型 API。请在下方填入您的 API Key 和 Base URL。本代码默认支持通义千问 (DashScope) 与硅基流动 (SiliconFlow)，也可以直接使用 OpenAI 或其他兼容的 API 接口。

In [ ]:
import os
import json
import time
import httpx
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（也可以直接读取系统环境变量）
# ============================================================
API_KEY = ""         # 例如: "sk-abc123..."
BASE_URL = "https://api.siliconflow.cn/v1"        # 例如: "https://api.siliconflow.cn/v1" 或 "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V3"      # 例如: "deepseek-ai/DeepSeek-V3" 或 "qwen-plus"

# 优先从上面填写的变量读取，其次读取系统环境变量
api_key = API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("SILICONFLOW_API_KEY") or os.environ.get("DASH_SCOPE_API_KEY")
base_url = BASE_URL or os.environ.get("OPENAI_API_BASE")
model_name = MODEL_NAME or os.environ.get("OPENAI_API_MODEL")

# 自动识别常见服务商的环境变量
if not base_url:
    if os.environ.get("SILICONFLOW_API_KEY"):
        base_url = "https://api.siliconflow.cn/v1"
        model_name = model_name or "deepseek-ai/DeepSeek-V3"
    elif os.environ.get("DASH_SCOPE_API_KEY"):
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        model_name = model_name or "qwen-plus"
    else:
        base_url = "https://api.openai.com/v1"
        model_name = model_name or "gpt-4o-mini"

if not api_key:
    raise ValueError("❌ 未检测到 API Key！请在上方单元格中配置您的 API Key 或设置系统环境变量。")

# 💡 如果您在 Windows 环境下使用 VPN 或代理工具，可能会遇到 SSL/ConnectError 报错。
# 此时可以尝试取消下方这行代码的注释，使用禁用了 SSL 证书验证的自定义 httpx 客户端：
# openai_client = OpenAI(api_key=api_key, base_url=base_url, http_client=httpx.Client(verify=False))
openai_client = OpenAI(api_key=api_key, base_url=base_url)
print(f"✨ 客户端实例化成功！当前使用的接口地址为: {base_url}，模型为: {model_name}")

## 📂 测试数据生成

为了进行文件查找实验，我们需要在本地创建一个测试目录 `data/fc_test`，并在其中放入几个测试文件，包括我们要让大模型查找的目标文件 `target_simple.txt`。

In [ ]:
import shutil

# 创建测试目录
test_dir = "data/fc_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir, exist_ok=True)

# 写入一些干扰文件与目标文件
files_to_create = {
    "report.txt": "2026年年度财务报告：公司业绩稳步上升。",
    "config.json": '{"port": 8080, "debug": false}',
    "target_simple.txt": "恭喜你！成功找到了 target_simple.txt 文件。密钥为: SIMPLE_SUCCESS_2026",
    "notes.log": "2026-06-03 12:00:00 - Server started successfully."
}

for filename, content in files_to_create.items():
    with open(os.path.join(test_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"💾 测试目录 {test_dir} 准备完毕，已创建 {len(files_to_create)} 个文件。")

## 🛡️ 在本地封装通用 CLI 命令执行与白名单过滤函数

为了模拟真实 Agent 灵活调用 CLI 的方式，我们坚持**不为特定任务定义专用的 Python 函数**（例如不定义 `run_cli_search` 这样的包装器）。相反，我们只提供一个通用的终端执行函数 `run_command(command: str)`，由大模型自己决策输入什么命令行指令。

同时，为了保证宿主机系统的绝对安全，我们在 `run_command` 内部对可调用的命令进行严格限制，**仅允许执行安全白名单命令（如 `pwd`、`ls`、`grep`、`cat`、`echo`）**。另外，考虑到学生们可能在 Windows 等不同平台上运行，我们在本地对这些命令进行了跨平台适配与仿真。

In [ ]:
import subprocess
import sys
import os
import shlex

# 定义安全命令白名单
ALLOWED_COMMANDS = {"pwd", "ls", "grep", "cat", "echo"}

def run_command(command: str) -> str:
    """
    通用命令执行器：执行终端命令，但仅限安全白名单命令，支持跨平台仿真。
    """
    print(f"🔧 [OS 进程执行] 收到待执行命令行: {command}")
    try:
        # 1. 安全解析命令行参数，避免 Shell 注入
        args = shlex.split(command)
        if not args:
            return "错误：命令不能为空。"
        
        base_cmd = args[0]
        
        # 2. 白名单检查
        if base_cmd not in ALLOWED_COMMANDS:
            return f"⚠️ 安全警告：非法的命令调用！该环境仅允许运行以下安全命令: {list(ALLOWED_COMMANDS)}"
        
        # 3. 针对 Windows 环境进行常用命令仿真，防止 executable 找不到报错
        if os.name == 'nt':
            if base_cmd == "pwd":
                return os.getcwd()
            elif base_cmd == "ls":
                path = args[1] if len(args) > 1 else "."
                if os.path.exists(path):
                    return "\n".join(os.listdir(path))
                return f"ls: {path}: No such file or directory"
            elif base_cmd == "cat":
                if len(args) < 2:
                    return "cat: missing operand"
                path = args[1]
                if os.path.exists(path):
                    with open(path, 'r', encoding='utf-8') as f:
                        return f.read()
                return f"cat: {path}: No such file or directory"
            elif base_cmd == "echo":
                return " ".join(args[1:])
            elif base_cmd == "grep":
                if len(args) < 3:
                    return "grep: format is 'grep <pattern> <file>'"
                pattern, filepath = args[1], args[2]
                if os.path.exists(filepath):
                    with open(filepath, 'r', encoding='utf-8') as f:
                        lines = f.readlines()
                    matched = [line.strip() for line in lines if pattern in line]
                    return "\n".join(matched)
                return f"grep: {filepath}: No such file or directory"
        
        # 4. 类 Unix 系统下，直接拉起子进程执行，禁用 shell=True 以确保安全
        result = subprocess.run(
            args,
            capture_output=True,
            text=True,
            encoding='utf-8',
            check=False
        )
        return result.stdout.strip() if result.returncode == 0 else f"CLI 执行失败: {result.stderr.strip()}"
    except Exception as e:
        return f"运行时异常: {str(e)}"

# 验证通用受控命令行工具
print("🧪 验证合规命令 ls:")
print(run_command("ls data/fc_test"))
print("\n🧪 验证合规命令 cat:")
print(run_command("cat data/fc_test/target_simple.txt"))
print("\n🧪 验证违规命令 whoami:")
print(run_command("whoami"))

## 🤖 向大模型注册 CLI 工具并实现端到端 Agent 检索

我们将通用命令执行器 `run_command` 描述以 Schema 形式发送给大模型，让大模型自主决策组合生成命令行指令来查找和读取文件。

In [ ]:
cli_tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "在本地终端执行系统命令行指令。为了安全，仅允许运行 ['pwd', 'ls', 'grep', 'cat', 'echo'] 这些安全白名单命令。你可以自由组合使用这些命令完成文件查找与内容读取任务。",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "待执行的终端命令，例如 'ls data/fc_test' 或 'cat data/fc_test/target_simple.txt'"
                    }
                },
                "required": ["command"]
            }
        }
    }
]

user_query = "我想知道 data/fc_test 目录下有没有 target_simple.txt 这个文件，它的内容是什么？"
messages = [{"role": "user", "content": user_query}]

step = 0
max_steps = 5

while step < max_steps:
    print(f"\n--- 🔄 第 {step + 1} 轮大模型推理中... ---")
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=cli_tool_schema,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    messages.append(assistant_message)
    
    # 检查大模型是否需要调用工具
    tool_calls = assistant_message.tool_calls
    if not tool_calls:
        print("\n🏆 大模型已获得所有信息，给出最终自然语言答复:")
        print("=" * 65)
        print(assistant_message.content)
        print("=" * 65)
        break
        
    print("🤖 大模型决策结果: 【需要调用外部工具】")
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        if func_name == "run_command":
            # 执行通用 CLI 命令安全受控调用
            cli_output = run_command(func_args.get("command"))
            print(f"   📥 [CLI 响应结果]:\n{cli_output}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": cli_output
            })
            
    step += 1
else:
    print("⚠️ 思考轮数达到上限，防止死循环。")